<style>
/* =====================================================================
   persian-rtl-jupyter.css
   هدف: نمایش درست متن فارسی (راست‌به‌چپ) همراه با فرمول‌های ریاضی
   (MathJax) و کد/متن انگلیسی (چپ‌به‌راست) در سلول‌های Markdown نوت‌بوک
   جوپیتر.

   این فایل برای سه محیط پوشش داده شده:
     1) JupyterLab (کلاس‌های .jp-*)
     2) Notebook کلاسیک (کلاس‌های .text_cell_render / .rendered_html)
     3) خروجی nbconvert به HTML و VS Code Jupyter (.rendered_html)
   ===================================================================== */

/* --- 1) جهت پایه‌ی سلول‌های مارک‌داون: راست‌به‌چپ ---
   معادل عملکرد dir="rtl": direction + unicode-bidi:isolate روی کانتینر،
   و direction:rtl روی *همه‌ی* فرزندان (strong, em, li, ...) تا هیچ تگی
   (مثل **بولد**) استایل را گم نکند. */
.jp-MarkdownCell .jp-RenderedMarkdown,
.text_cell_render,
.rendered_html {
    direction: rtl;
    text-align: right;
    unicode-bidi: isolate;
    font-family: "Vazirmatn", "Vazir", "Tahoma", "IRANSans", sans-serif;
    line-height: 1.9;
}

.jp-MarkdownCell .jp-RenderedMarkdown *,
.text_cell_render *,
.rendered_html * {
    direction: rtl;
    unicode-bidi: isolate;
}

/* --- 2) فرمول‌های ریاضی: همیشه چپ‌به‌راست و مستقل از متن اطراف --- */
.jp-MarkdownCell .jp-RenderedMarkdown mjx-container,
.text_cell_render mjx-container,
.rendered_html mjx-container,
.jp-MarkdownCell .jp-RenderedMarkdown .MathJax,
.text_cell_render .MathJax,
.rendered_html .MathJax,
.jp-MarkdownCell .jp-RenderedMarkdown .MathJax_Display,
.text_cell_render .MathJax_Display,
.rendered_html .MathJax_Display {
    direction: ltr;
    unicode-bidi: isolate;               /* فرمول یک "جزیره‌ی" مستقل LTR می‌شود */
    text-align: left;
}

/* فرمول‌های نمایشی (روی خط جدا، $$...$$) وسط‌چین بمانند */
mjx-container[display="true"] {
    text-align: center !important;
}

/* --- 3) کد inline و بلوک‌های کد: چپ‌به‌راست --- */
.jp-MarkdownCell .jp-RenderedMarkdown code,
.jp-MarkdownCell .jp-RenderedMarkdown pre,
.text_cell_render code,
.text_cell_render pre,
.rendered_html code,
.rendered_html pre {
    direction: ltr;
    unicode-bidi: isolate;
    text-align: left;
    font-family: "Fira Code", "Consolas", monospace;
}

/* --- 4) لیست‌ها و جدول‌ها هم راست‌چین شوند --- */
.jp-MarkdownCell .jp-RenderedMarkdown ul,
.jp-MarkdownCell .jp-RenderedMarkdown ol,
.text_cell_render ul,
.text_cell_render ol,
.rendered_html ul,
.rendered_html ol {
    padding-right: 1.5em;
    padding-left: 0;
}

.jp-MarkdownCell .jp-RenderedMarkdown table,
.text_cell_render table,
.rendered_html table {
    direction: rtl;
}
</style>
<div dir=rtl>




# جلسه ۲ — تفکر تانسوری و برداری در پایتون (Broadcasting)
### مبانی ریاضی یادگیری ماشین | بخش اول: مقدمه و پایه‌گذاری رویکرد

**اهداف یادگیری این جلسه:**
- یادگیری اینکه چرا در برنامه‌نویسی داده‌محور، باید به‌جای «حلقه زدن روی تک‌تک اعداد»، به زبان **بردار و ماتریس** فکر کنیم
- آشنایی با ساخت و کار با تانسورها (اسکالر، بردار، ماتریس، تانسور n‑بعدی) در NumPy و PyTorch
- یادگیری **Broadcasting**: قانونی که به ما اجازه می‌دهد آرایه‌هایی با شکل‌های متفاوت را بدون نوشتن حلقه، با هم ترکیب کنیم
- توانایی خواندن و رفع خطاهای رایج ناشی از عدم تطابق شکل (shape mismatch)

> 📘 **جعبه واژه‌نامه** — هر جا اصطلاح تخصصی جدیدی به‌کار می‌رود، همین‌جا با یک مثال ساده و غیرتخصصی توضیح داده می‌شود. لازم نیست از قبل این کلمات را بلد باشید.

> **مطالعه‌ی این جلسه:** فصل ۲ کتاب *Dive into Deep Learning* (بخش‌های Data Manipulation و Broadcasting). اگر NumPy یا PyTorch برایتان کاملاً تازه است، نگران نباشید — این جلسه از صفر شروع می‌کند.


<style>
/* =====================================================================
   persian-rtl-jupyter.css
   هدف: نمایش درست متن فارسی (راست‌به‌چپ) همراه با فرمول‌های ریاضی
   (MathJax) و کد/متن انگلیسی (چپ‌به‌راست) در سلول‌های Markdown نوت‌بوک
   جوپیتر.

   این فایل برای سه محیط پوشش داده شده:
     1) JupyterLab (کلاس‌های .jp-*)
     2) Notebook کلاسیک (کلاس‌های .text_cell_render / .rendered_html)
     3) خروجی nbconvert به HTML و VS Code Jupyter (.rendered_html)
   ===================================================================== */

/* --- 1) جهت پایه‌ی سلول‌های مارک‌داون: راست‌به‌چپ ---
   معادل عملکرد dir="rtl": direction + unicode-bidi:isolate روی کانتینر،
   و direction:rtl روی *همه‌ی* فرزندان (strong, em, li, ...) تا هیچ تگی
   (مثل **بولد**) استایل را گم نکند. */
.jp-MarkdownCell .jp-RenderedMarkdown,
.text_cell_render,
.rendered_html {
    direction: rtl;
    text-align: right;
    unicode-bidi: isolate;
    font-family: "Vazirmatn", "Vazir", "Tahoma", "IRANSans", sans-serif;
    line-height: 1.9;
}

.jp-MarkdownCell .jp-RenderedMarkdown *,
.text_cell_render *,
.rendered_html * {
    direction: rtl;
    unicode-bidi: isolate;
}

/* --- 2) فرمول‌های ریاضی: همیشه چپ‌به‌راست و مستقل از متن اطراف --- */
.jp-MarkdownCell .jp-RenderedMarkdown mjx-container,
.text_cell_render mjx-container,
.rendered_html mjx-container,
.jp-MarkdownCell .jp-RenderedMarkdown .MathJax,
.text_cell_render .MathJax,
.rendered_html .MathJax,
.jp-MarkdownCell .jp-RenderedMarkdown .MathJax_Display,
.text_cell_render .MathJax_Display,
.rendered_html .MathJax_Display {
    direction: ltr;
    unicode-bidi: isolate;               /* فرمول یک "جزیره‌ی" مستقل LTR می‌شود */
    text-align: left;
}

/* فرمول‌های نمایشی (روی خط جدا، $$...$$) وسط‌چین بمانند */
mjx-container[display="true"] {
    text-align: center !important;
}

/* --- 3) کد inline و بلوک‌های کد: چپ‌به‌راست --- */
.jp-MarkdownCell .jp-RenderedMarkdown code,
.jp-MarkdownCell .jp-RenderedMarkdown pre,
.text_cell_render code,
.text_cell_render pre,
.rendered_html code,
.rendered_html pre {
    direction: ltr;
    unicode-bidi: isolate;
    text-align: left;
    font-family: "Fira Code", "Consolas", monospace;
}

/* --- 4) لیست‌ها و جدول‌ها هم راست‌چین شوند --- */
.jp-MarkdownCell .jp-RenderedMarkdown ul,
.jp-MarkdownCell .jp-RenderedMarkdown ol,
.text_cell_render ul,
.text_cell_render ol,
.rendered_html ul,
.rendered_html ol {
    padding-right: 1.5em;
    padding-left: 0;
}

.jp-MarkdownCell .jp-RenderedMarkdown table,
.text_cell_render table,
.rendered_html table {
    direction: rtl;
}
</style>
<div dir=rtl>


## 🎯 چالش واقعی این جلسه: چرا فروشگاه آنلاین شما با یک حلقه‌ی ساده کند می‌شود؟

فرض کنید فروشگاه آنلاین شما ۱ میلیون محصول دارد و می‌خواهید قیمت همه را ۱۰٪ افزایش دهید. ساده‌ترین راهی که به ذهن می‌رسد این است: یک حلقه بنویسیم که روی تک‌تک محصولات برود و قیمتش را عوض کند.

بیایید ببینیم این روش چقدر طول می‌کشد، و بعد یک راه جایگزین را امتحان کنیم.

In [3]:
import numpy as np
import torch
import time

np.random.seed(0)
n_products = 1_000_000
prices = np.random.uniform(10, 1000, size=n_products)
prices_list = prices.tolist()  # همان داده، ولی به شکل یک لیست پایتونی معمولی

# روش ۱: حلقه‌ی معمولی پایتون
start = time.time()
new_prices_loop = [p * 1.10 for p in prices_list]
loop_time = time.time() - start
print(f"⏱️ زمان روش حلقه‌ای: {loop_time:.4f} ثانیه")


⏱️ زمان روش حلقه‌ای: 0.0577 ثانیه


In [4]:
# روش ۲: تفکر برداری (Vectorization) — کل آرایه را یک‌جا ضرب می‌کنیم، بدون حلقه‌ی صریح
start = time.time()
new_prices_vectorized = prices * 1.10
vectorized_time = time.time() - start
print(f"⏱️ زمان روش برداری: {vectorized_time:.4f} ثانیه")

print(f"\n🚀 روش برداری حدود {loop_time / vectorized_time:.1f} برابر سریع‌تر بود!")
print("نتیجه‌ی هر دو روش یکسان است؟", np.allclose(new_prices_loop, new_prices_vectorized))


⏱️ زمان روش برداری: 0.0056 ثانیه

🚀 روش برداری حدود 10.3 برابر سریع‌تر بود!
نتیجه‌ی هر دو روش یکسان است؟ True


<style>
/* =====================================================================
   persian-rtl-jupyter.css
   هدف: نمایش درست متن فارسی (راست‌به‌چپ) همراه با فرمول‌های ریاضی
   (MathJax) و کد/متن انگلیسی (چپ‌به‌راست) در سلول‌های Markdown نوت‌بوک
   جوپیتر.

   این فایل برای سه محیط پوشش داده شده:
     1) JupyterLab (کلاس‌های .jp-*)
     2) Notebook کلاسیک (کلاس‌های .text_cell_render / .rendered_html)
     3) خروجی nbconvert به HTML و VS Code Jupyter (.rendered_html)
   ===================================================================== */

/* --- 1) جهت پایه‌ی سلول‌های مارک‌داون: راست‌به‌چپ ---
   معادل عملکرد dir="rtl": direction + unicode-bidi:isolate روی کانتینر،
   و direction:rtl روی *همه‌ی* فرزندان (strong, em, li, ...) تا هیچ تگی
   (مثل **بولد**) استایل را گم نکند. */
.jp-MarkdownCell .jp-RenderedMarkdown,
.text_cell_render,
.rendered_html {
    direction: rtl;
    text-align: right;
    unicode-bidi: isolate;
    font-family: "Vazirmatn", "Vazir", "Tahoma", "IRANSans", sans-serif;
    line-height: 1.9;
}

.jp-MarkdownCell .jp-RenderedMarkdown *,
.text_cell_render *,
.rendered_html * {
    direction: rtl;
    unicode-bidi: isolate;
}

/* --- 2) فرمول‌های ریاضی: همیشه چپ‌به‌راست و مستقل از متن اطراف --- */
.jp-MarkdownCell .jp-RenderedMarkdown mjx-container,
.text_cell_render mjx-container,
.rendered_html mjx-container,
.jp-MarkdownCell .jp-RenderedMarkdown .MathJax,
.text_cell_render .MathJax,
.rendered_html .MathJax,
.jp-MarkdownCell .jp-RenderedMarkdown .MathJax_Display,
.text_cell_render .MathJax_Display,
.rendered_html .MathJax_Display {
    direction: ltr;
    unicode-bidi: isolate;               /* فرمول یک "جزیره‌ی" مستقل LTR می‌شود */
    text-align: left;
}

/* فرمول‌های نمایشی (روی خط جدا، $$...$$) وسط‌چین بمانند */
mjx-container[display="true"] {
    text-align: center !important;
}

/* --- 3) کد inline و بلوک‌های کد: چپ‌به‌راست --- */
.jp-MarkdownCell .jp-RenderedMarkdown code,
.jp-MarkdownCell .jp-RenderedMarkdown pre,
.text_cell_render code,
.text_cell_render pre,
.rendered_html code,
.rendered_html pre {
    direction: ltr;
    unicode-bidi: isolate;
    text-align: left;
    font-family: "Fira Code", "Consolas", monospace;
}

/* --- 4) لیست‌ها و جدول‌ها هم راست‌چین شوند --- */
.jp-MarkdownCell .jp-RenderedMarkdown ul,
.jp-MarkdownCell .jp-RenderedMarkdown ol,
.text_cell_render ul,
.text_cell_render ol,
.rendered_html ul,
.rendered_html ol {
    padding-right: 1.5em;
    padding-left: 0;
}

.jp-MarkdownCell .jp-RenderedMarkdown table,
.text_cell_render table,
.rendered_html table {
    direction: rtl;
}
</style>
<div dir=rtl>


📘 به این‌که یک عملیات را روی **کل آرایه** یک‌جا انجام دهیم، به‌جای اینکه با حلقه روی تک‌تک عضوهایش برویم، **برداری‌سازی (Vectorization)** می‌گویند. NumPy و PyTorch این عملیات را در پس‌زمینه با کد بسیار بهینه‌شده (نوشته‌شده به زبان C) انجام می‌دهند — به همین دلیل خیلی سریع‌تر از حلقه‌ی معمولی پایتون است.

**این  همان چیزی است که در  این درس دنبالش هستیم:** یاد بگیریم به‌جای «تک‌تک اعداد»، به زبان **بردار، ماتریس و تانسور** فکر کنیم — هم برای سرعت، و هم چون فرمول‌های ریاضی یادگیری ماشین دقیقاً به همین زبان نوشته می‌شوند.

<style>
/* =====================================================================
   persian-rtl-jupyter.css
   هدف: نمایش درست متن فارسی (راست‌به‌چپ) همراه با فرمول‌های ریاضی
   (MathJax) و کد/متن انگلیسی (چپ‌به‌راست) در سلول‌های Markdown نوت‌بوک
   جوپیتر.

   این فایل برای سه محیط پوشش داده شده:
     1) JupyterLab (کلاس‌های .jp-*)
     2) Notebook کلاسیک (کلاس‌های .text_cell_render / .rendered_html)
     3) خروجی nbconvert به HTML و VS Code Jupyter (.rendered_html)
   ===================================================================== */

/* --- 1) جهت پایه‌ی سلول‌های مارک‌داون: راست‌به‌چپ ---
   معادل عملکرد dir="rtl": direction + unicode-bidi:isolate روی کانتینر،
   و direction:rtl روی *همه‌ی* فرزندان (strong, em, li, ...) تا هیچ تگی
   (مثل **بولد**) استایل را گم نکند. */
.jp-MarkdownCell .jp-RenderedMarkdown,
.text_cell_render,
.rendered_html {
    direction: rtl;
    text-align: right;
    unicode-bidi: isolate;
    font-family: "Vazirmatn", "Vazir", "Tahoma", "IRANSans", sans-serif;
    line-height: 1.9;
}

.jp-MarkdownCell .jp-RenderedMarkdown *,
.text_cell_render *,
.rendered_html * {
    direction: rtl;
    unicode-bidi: isolate;
}

/* --- 2) فرمول‌های ریاضی: همیشه چپ‌به‌راست و مستقل از متن اطراف --- */
.jp-MarkdownCell .jp-RenderedMarkdown mjx-container,
.text_cell_render mjx-container,
.rendered_html mjx-container,
.jp-MarkdownCell .jp-RenderedMarkdown .MathJax,
.text_cell_render .MathJax,
.rendered_html .MathJax,
.jp-MarkdownCell .jp-RenderedMarkdown .MathJax_Display,
.text_cell_render .MathJax_Display,
.rendered_html .MathJax_Display {
    direction: ltr;
    unicode-bidi: isolate;               /* فرمول یک "جزیره‌ی" مستقل LTR می‌شود */
    text-align: left;
}

/* فرمول‌های نمایشی (روی خط جدا، $$...$$) وسط‌چین بمانند */
mjx-container[display="true"] {
    text-align: center !important;
}

/* --- 3) کد inline و بلوک‌های کد: چپ‌به‌راست --- */
.jp-MarkdownCell .jp-RenderedMarkdown code,
.jp-MarkdownCell .jp-RenderedMarkdown pre,
.text_cell_render code,
.text_cell_render pre,
.rendered_html code,
.rendered_html pre {
    direction: ltr;
    unicode-bidi: isolate;
    text-align: left;
    font-family: "Fira Code", "Consolas", monospace;
}

/* --- 4) لیست‌ها و جدول‌ها هم راست‌چین شوند --- */
.jp-MarkdownCell .jp-RenderedMarkdown ul,
.jp-MarkdownCell .jp-RenderedMarkdown ol,
.text_cell_render ul,
.text_cell_render ol,
.rendered_html ul,
.rendered_html ol {
    padding-right: 1.5em;
    padding-left: 0;
}

.jp-MarkdownCell .jp-RenderedMarkdown table,
.text_cell_render table,
.rendered_html table {
    direction: rtl;
}
</style>
<div dir=rtl>


### 🗣️ بحث دو‌نفره (Think–Pair–Share) — ۲ دقیقه
با نفر کناری‌تان فکر کنید: اگر فروشگاه شما به‌جای ۱ میلیون محصول، ۱ میلیارد محصول داشت، فکر می‌کنید این تفاوت سرعت بیشتر می‌شد یا کمتر؟ چرا؟

<style>
/* =====================================================================
   persian-rtl-jupyter.css
   هدف: نمایش درست متن فارسی (راست‌به‌چپ) همراه با فرمول‌های ریاضی
   (MathJax) و کد/متن انگلیسی (چپ‌به‌راست) در سلول‌های Markdown نوت‌بوک
   جوپیتر.

   این فایل برای سه محیط پوشش داده شده:
     1) JupyterLab (کلاس‌های .jp-*)
     2) Notebook کلاسیک (کلاس‌های .text_cell_render / .rendered_html)
     3) خروجی nbconvert به HTML و VS Code Jupyter (.rendered_html)
   ===================================================================== */

/* --- 1) جهت پایه‌ی سلول‌های مارک‌داون: راست‌به‌چپ ---
   معادل عملکرد dir="rtl": direction + unicode-bidi:isolate روی کانتینر،
   و direction:rtl روی *همه‌ی* فرزندان (strong, em, li, ...) تا هیچ تگی
   (مثل **بولد**) استایل را گم نکند. */
.jp-MarkdownCell .jp-RenderedMarkdown,
.text_cell_render,
.rendered_html {
    direction: rtl;
    text-align: right;
    unicode-bidi: isolate;
    font-family: "Vazirmatn", "Vazir", "Tahoma", "IRANSans", sans-serif;
    line-height: 1.9;
}

.jp-MarkdownCell .jp-RenderedMarkdown *,
.text_cell_render *,
.rendered_html * {
    direction: rtl;
    unicode-bidi: isolate;
}

/* --- 2) فرمول‌های ریاضی: همیشه چپ‌به‌راست و مستقل از متن اطراف --- */
.jp-MarkdownCell .jp-RenderedMarkdown mjx-container,
.text_cell_render mjx-container,
.rendered_html mjx-container,
.jp-MarkdownCell .jp-RenderedMarkdown .MathJax,
.text_cell_render .MathJax,
.rendered_html .MathJax,
.jp-MarkdownCell .jp-RenderedMarkdown .MathJax_Display,
.text_cell_render .MathJax_Display,
.rendered_html .MathJax_Display {
    direction: ltr;
    unicode-bidi: isolate;               /* فرمول یک "جزیره‌ی" مستقل LTR می‌شود */
    text-align: left;
}

/* فرمول‌های نمایشی (روی خط جدا، $$...$$) وسط‌چین بمانند */
mjx-container[display="true"] {
    text-align: center !important;
}

/* --- 3) کد inline و بلوک‌های کد: چپ‌به‌راست --- */
.jp-MarkdownCell .jp-RenderedMarkdown code,
.jp-MarkdownCell .jp-RenderedMarkdown pre,
.text_cell_render code,
.text_cell_render pre,
.rendered_html code,
.rendered_html pre {
    direction: ltr;
    unicode-bidi: isolate;
    text-align: left;
    font-family: "Fira Code", "Consolas", monospace;
}

/* --- 4) لیست‌ها و جدول‌ها هم راست‌چین شوند --- */
.jp-MarkdownCell .jp-RenderedMarkdown ul,
.jp-MarkdownCell .jp-RenderedMarkdown ol,
.text_cell_render ul,
.text_cell_render ol,
.rendered_html ul,
.rendered_html ol {
    padding-right: 1.5em;
    padding-left: 0;
}

.jp-MarkdownCell .jp-RenderedMarkdown table,
.text_cell_render table,
.rendered_html table {
    direction: rtl;
}
</style>
<div dir=rtl>



## بخش ۱: از عدد تا تانسور — ساختارهای داده‌ای پایه

📘 در برنامه‌نویسی علمی، داده را در سطوح مختلفی سازمان‌دهی می‌کنیم:

| نام | رتبه (rank) | مثال دنیای واقعی |
|---|---|---|
| **اسکالر (Scalar)** | ۰ | یک عدد تنها، مثلاً قیمت یک محصول: `29.99` |
| **بردار (Vector)** | ۱ | ویژگی‌های یک کاربر: `[سن, درآمد, تعداد خرید]` |
| **ماتریس (Matrix)** | ۲ | جدولی از چند کاربر × چند ویژگی |
| **تانسور (Tensor)** | ≥۳ | یک دسته (batch) از تصاویر رنگی، یا چند جدول کنار هم |

به تعداد بُعدهای یک آرایه، **رتبه (rank)** یا **ابعاد (dimensions)** می‌گویند، و به اندازه‌ی هر بُعد، **شکل (shape)** آن آرایه گفته می‌شود. یکی از رایج‌ترین خطاهایی که در کدنویسی یادگیری ماشین رخ می‌دهد، ناهماهنگی shape است — پس از همین امروز عادت کنید همیشه shape داده‌تان را چک کنید.

<style>
/* =====================================================================
   persian-rtl-jupyter.css
   هدف: نمایش درست متن فارسی (راست‌به‌چپ) همراه با فرمول‌های ریاضی
   (MathJax) و کد/متن انگلیسی (چپ‌به‌راست) در سلول‌های Markdown نوت‌بوک
   جوپیتر.

   این فایل برای سه محیط پوشش داده شده:
     1) JupyterLab (کلاس‌های .jp-*)
     2) Notebook کلاسیک (کلاس‌های .text_cell_render / .rendered_html)
     3) خروجی nbconvert به HTML و VS Code Jupyter (.rendered_html)
   ===================================================================== */

/* --- 1) جهت پایه‌ی سلول‌های مارک‌داون: راست‌به‌چپ ---
   معادل عملکرد dir="rtl": direction + unicode-bidi:isolate روی کانتینر،
   و direction:rtl روی *همه‌ی* فرزندان (strong, em, li, ...) تا هیچ تگی
   (مثل **بولد**) استایل را گم نکند. */
.jp-MarkdownCell .jp-RenderedMarkdown,
.text_cell_render,
.rendered_html {
    direction: rtl;
    text-align: right;
    unicode-bidi: isolate;
    font-family: "Vazirmatn", "Vazir", "Tahoma", "IRANSans", sans-serif;
    line-height: 1.9;
}

.jp-MarkdownCell .jp-RenderedMarkdown *,
.text_cell_render *,
.rendered_html * {
    direction: rtl;
    unicode-bidi: isolate;
}

/* --- 2) فرمول‌های ریاضی: همیشه چپ‌به‌راست و مستقل از متن اطراف --- */
.jp-MarkdownCell .jp-RenderedMarkdown mjx-container,
.text_cell_render mjx-container,
.rendered_html mjx-container,
.jp-MarkdownCell .jp-RenderedMarkdown .MathJax,
.text_cell_render .MathJax,
.rendered_html .MathJax,
.jp-MarkdownCell .jp-RenderedMarkdown .MathJax_Display,
.text_cell_render .MathJax_Display,
.rendered_html .MathJax_Display {
    direction: ltr;
    unicode-bidi: isolate;               /* فرمول یک "جزیره‌ی" مستقل LTR می‌شود */
    text-align: left;
}

/* فرمول‌های نمایشی (روی خط جدا، $$...$$) وسط‌چین بمانند */
mjx-container[display="true"] {
    text-align: center !important;
}

/* --- 3) کد inline و بلوک‌های کد: چپ‌به‌راست --- */
.jp-MarkdownCell .jp-RenderedMarkdown code,
.jp-MarkdownCell .jp-RenderedMarkdown pre,
.text_cell_render code,
.text_cell_render pre,
.rendered_html code,
.rendered_html pre {
    direction: ltr;
    text-align: left;
    font-family: "Fira Code", "Consolas", monospace;
}

/* --- 4) لیست‌ها و جدول‌ها هم راست‌چین شوند --- */
.jp-MarkdownCell .jp-RenderedMarkdown ul,
.jp-MarkdownCell .jp-RenderedMarkdown ol,
.text_cell_render ul,
.text_cell_render ol,
.rendered_html ul,
.rendered_html ol {
    padding-right: 1.5em;
    padding-left: 0;
}

.jp-MarkdownCell .jp-RenderedMarkdown table,
.text_cell_render table,
.rendered_html table {
    direction: rtl;
}
</style>
<div dir=rtl>

# Tensor چیست؟

##  تعریف دقیق ریاضی

به‌طور دقیق، یک Tensor را می‌توان به‌صورت یک شیء  چندبعدی در نظر گرفت. در کاربردهای یادگیری ماشین، معمولاً با نمایش مؤلفه‌ای Tensor سر و کار داریم.

یک Tensor مرتبه‌ی $k$ روی میدان $\mathbb{R}$ را می‌توان به‌صورت آرایه‌ای از اعداد حقیقی با $k$ اندیس تعریف کرد:

$$
\mathcal{T}
=
\left(T_{i_1,i_2,\ldots,i_k}\right)
$$

که در آن

$$
1\leq i_j\leq n_j,
\qquad
j=1,\ldots,k.
$$

بنابراین:

$$
\mathcal{T}\in
\mathbb{R}^{n_1\times n_2\times\cdots\times n_k}.
$$

به تعداد اندیس‌های Tensor، معمولاً **مرتبه** یا **Order** آن گفته می‌شود:

$$
\operatorname{order}(\mathcal{T})=k.
$$

مثلاً:

$$
\mathcal{T}\in\mathbb{R}^{3}
$$

یک Tensor مرتبه‌ی $1$ است،

$$
A\in\mathbb{R}^{m\times n}
$$

یک Tensor مرتبه‌ی $2$ است، و

$$
\mathcal{X}\in
\mathbb{R}^{n_1\times n_2\times n_3}
$$

یک Tensor مرتبه‌ی $3$ است.

---

##  رابطه‌ی Tensor با اسکالر، بردار و ماتریس

می‌توانیم یک سلسله‌مراتب ساده داشته باشیم:

$$
\boxed{
\text{Scalar}
\longrightarrow
\text{Vector}
\longrightarrow
\text{Matrix}
\longrightarrow
\text{Higher-order Tensor}
}
$$

یا:

$$
\mathbb{R}
$$

برای اسکالر،

$$
\mathbb{R}^{n}
$$

برای بردار،

$$
\mathbb{R}^{m\times n}
$$

برای ماتریس، و

$$
\mathbb{R}^{n_1\times n_2\times n_3}
$$

برای Tensor مرتبه‌ی $3$.

بنابراین در کاربردهای معمول یادگیری ماشین، Tensor را می‌توان به‌صورت تعمیم ماتریس به تعداد بیشتری از ابعاد تصور کرد.

---

##  مثال ساده

فرض کنید:

$$
x=
\begin{bmatrix}
1&2&3
\end{bmatrix}.
$$

این یک Tensor مرتبه‌ی $1$ با شکل

$$
\operatorname{shape}(x)=(3)
$$

است.

یک ماتریس مانند

$$
A=
\begin{bmatrix}
1&2&3\\
4&5&6
\end{bmatrix}
$$

یک Tensor مرتبه‌ی $2$ با شکل

$$
\operatorname{shape}(A)=(2,3)
$$

است.

حالا فرض کنید سه ماتریس $2\times3$ داشته باشیم:

$$
A_1=
\begin{bmatrix}
1&2&3\\
4&5&6
\end{bmatrix},
\qquad
A_2=
\begin{bmatrix}
7&8&9\\
10&11&12
\end{bmatrix},
$$

و

$$
A_3=
\begin{bmatrix}
13&14&15\\
16&17&18
\end{bmatrix}.
$$

اگر این سه ماتریس را در یک ساختار قرار دهیم، یک Tensor مرتبه‌ی $3$ خواهیم داشت:

$$
\mathcal{X}\in\mathbb{R}^{3\times2\times3}.
$$

در اینجا:

$$
\operatorname{shape}(\mathcal{X})=(3,2,3).
$$

می‌توانیم آن را به‌صورت سه لایه‌ی $2\times3$ تصور کنیم:

$$
\mathcal{X}_{1,:,:}=A_1,
\qquad
\mathcal{X}_{2,:,:}=A_2,
\qquad
\mathcal{X}_{3,:,:}=A_3.
$$

---

#  Tensor در Machine Learning

در یادگیری ماشین، داده‌ها معمولاً به شکل Tensor ذخیره می‌شوند، زیرا داده‌ها اغلب بیش از دو بُعد دارند.

##  داده‌های جدولی

فرض کنید $100$ نمونه داشته باشیم و از هر نمونه $5$ ویژگی استخراج کرده باشیم.

داده‌ها را می‌توان به شکل ماتریس زیر نمایش داد:

$$
X\in\mathbb{R}^{100\times5}.
$$

در اینجا:

$$
X_{ij}
$$

مقدار ویژگی $j$ برای نمونه‌ی $i$ است.

این یک Tensor مرتبه‌ی $2$ است.

---

##  تصویر

فرض کنید یک تصویر رنگی با ارتفاع $224$، عرض $224$ و سه کانال رنگی داشته باشیم.

می‌توانیم آن را به شکل زیر نمایش دهیم:

$$
X\in\mathbb{R}^{224\times224\times3}.
$$

سه بُعد می‌توانند به‌ترتیب نشان‌دهنده‌ی ارتفاع، عرض و کانال‌های رنگی باشند.

بنابراین یک تصویر رنگی یک Tensor مرتبه‌ی $3$ است.

برای مثال:

$$
X_{i,j,1}
$$

مقدار کانال اول در پیکسل واقع در موقعیت $(i,j)$ است.

---

##  مجموعه‌ای از تصاویر

حالا فرض کنید $32$ تصویر داشته باشیم و هر تصویر شکل

$$
224\times224\times3
$$

داشته باشد.

کل داده‌ها می‌توانند به شکل

$$
X\in
\mathbb{R}^{32\times224\times224\times3}
$$

ذخیره شوند.

پس:

$$
\operatorname{shape}(X)
=
(32,224,224,3).
$$

اینجا بُعد اول مربوط به تعداد تصاویر است.

بنابراین:

$$
X_{n,i,j,c}
$$

یعنی مقدار کانال $c$ در پیکسل $(i,j)$ از تصویر $n$.

---

#  Tensor در PyTorch

در PyTorch، کلاس اصلی برای نمایش چنین داده‌هایی

```python
torch.Tensor
```

است.

مثلاً:

```python
x = torch.tensor([1., 2., 3.])
```

یک Tensor با شکل

$$
(3)
$$

ایجاد می‌کند.

اگر بنویسیم:

```python
x.shape
```

نتیجه:

```python
torch.Size([3])
```

خواهد بود.

برای ماتریس:

```python
X = torch.tensor([
    [1., 2., 3.],
    [4., 5., 6.]
])
```

داریم:

$$
\operatorname{shape}(X)=(2,3).
$$

---

## نکته‌ی مهم درباره‌ی Tensor در Machine Learning

در یادگیری ماشین، وقتی می‌گوییم:

$$
X\in\mathbb{R}^{32\times224\times224\times3}
$$

نباید فقط به آن به‌عنوان «یک آرایه‌ی چهاربعدی» نگاه کنیم.

هر بُعد معمولاً یک معنای مشخص دارد.

برای مثال:

$$
\boxed{
32
\times
224
\times
224
\times
3
}
$$

که می‌تواند به‌ترتیب نشان‌دهنده‌ی

$$
\boxed{
\text{تعداد نمونه}
\times
\text{ارتفاع}
\times
\text{عرض}
\times
\text{کانال}
}
$$

باشد.

بنابراین مفهوم Tensor در ML فقط «چند عدد کنار هم» نیست؛ بلکه یک ساختار چندبعدی است که هر محور آن می‌تواند معنای خاصی در مسئله داشته باشد.

## ارتباط Tensor با Broadcasting

این دقیقاً همان جایی است که بحث Broadcasting اهمیت پیدا می‌کند.

اگر داشته باشیم:

$$
X\in\mathbb{R}^{32\times224\times224\times3}
$$

و یک Tensor داشته باشیم:

$$
b\in\mathbb{R}^{3},
$$

می‌توانیم بنویسیم:

$$
X+b.
$$

به کمک Broadcasting، بردار $b$ در تمام $32$ تصویر و تمام $224\times224$ موقعیت مکانی اعمال می‌شود.

یعنی اگر

$$
b=(b_1,b_2,b_3)
$$

باشد، برای هر پیکسل، مقدار متناظر با سه کانال به آن اضافه می‌شود.

این نوع عملیات در محاسبات شبکه‌های عصبی بسیار رایج است


In [5]:
scalar = torch.tensor(29.99)
vector = torch.tensor([25.0, 5_000_000.0, 3.0])          # سن، درآمد، تعداد خرید یک کاربر
matrix = torch.tensor([[25.0, 5_000_000.0, 3.0],
                        [31.0, 8_000_000.0, 1.0]])         # دو کاربر
tensor3d = torch.rand(4, 28, 28)                            # مثلاً ۴ تصویر خاکستری ۲۸×۲۸ پیکسل

for name, t in [("scalar", scalar), ("vector", vector), ("matrix", matrix), ("tensor3d", tensor3d)]:
    print(f"{name:10s} | shape: {tuple(t.shape)} | تعداد ابعاد (ndim): {t.ndim}")


scalar     | shape: () | تعداد ابعاد (ndim): 0
vector     | shape: (3,) | تعداد ابعاد (ndim): 1
matrix     | shape: (2, 3) | تعداد ابعاد (ndim): 2
tensor3d   | shape: (4, 28, 28) | تعداد ابعاد (ndim): 3


<style>
/* =====================================================================
   persian-rtl-jupyter.css
   هدف: نمایش درست متن فارسی (راست‌به‌چپ) همراه با فرمول‌های ریاضی
   (MathJax) و کد/متن انگلیسی (چپ‌به‌راست) در سلول‌های Markdown نوت‌بوک
   جوپیتر.

   این فایل برای سه محیط پوشش داده شده:
     1) JupyterLab (کلاس‌های .jp-*)
     2) Notebook کلاسیک (کلاس‌های .text_cell_render / .rendered_html)
     3) خروجی nbconvert به HTML و VS Code Jupyter (.rendered_html)
   ===================================================================== */

/* --- 1) جهت پایه‌ی سلول‌های مارک‌داون: راست‌به‌چپ ---
   معادل عملکرد dir="rtl": direction + unicode-bidi:isolate روی کانتینر،
   و direction:rtl روی *همه‌ی* فرزندان (strong, em, li, ...) تا هیچ تگی
   (مثل **بولد**) استایل را گم نکند. */
.jp-MarkdownCell .jp-RenderedMarkdown,
.text_cell_render,
.rendered_html {
    direction: rtl;
    text-align: right;
    unicode-bidi: isolate;
    font-family: "Vazirmatn", "Vazir", "Tahoma", "IRANSans", sans-serif;
    line-height: 1.9;
}

.jp-MarkdownCell .jp-RenderedMarkdown *,
.text_cell_render *,
.rendered_html * {
    direction: rtl;
    unicode-bidi: isolate;
}

/* --- 2) فرمول‌های ریاضی: همیشه چپ‌به‌راست و مستقل از متن اطراف --- */
.jp-MarkdownCell .jp-RenderedMarkdown mjx-container,
.text_cell_render mjx-container,
.rendered_html mjx-container,
.jp-MarkdownCell .jp-RenderedMarkdown .MathJax,
.text_cell_render .MathJax,
.rendered_html .MathJax,
.jp-MarkdownCell .jp-RenderedMarkdown .MathJax_Display,
.text_cell_render .MathJax_Display,
.rendered_html .MathJax_Display {
    direction: ltr;
    unicode-bidi: isolate;               /* فرمول یک "جزیره‌ی" مستقل LTR می‌شود */
    text-align: left;
}

/* فرمول‌های نمایشی (روی خط جدا، $$...$$) وسط‌چین بمانند */
mjx-container[display="true"] {
    text-align: center !important;
}

/* --- 3) کد inline و بلوک‌های کد: چپ‌به‌راست --- */
.jp-MarkdownCell .jp-RenderedMarkdown code,
.jp-MarkdownCell .jp-RenderedMarkdown pre,
.text_cell_render code,
.text_cell_render pre,
.rendered_html code,
.rendered_html pre {
    direction: ltr;
    unicode-bidi: isolate;
    text-align: left;
    font-family: "Fira Code", "Consolas", monospace;
}

/* --- 4) لیست‌ها و جدول‌ها هم راست‌چین شوند --- */
.jp-MarkdownCell .jp-RenderedMarkdown ul,
.jp-MarkdownCell .jp-RenderedMarkdown ol,
.text_cell_render ul,
.text_cell_render ol,
.rendered_html ul,
.rendered_html ol {
    padding-right: 1.5em;
    padding-left: 0;
}

.jp-MarkdownCell .jp-RenderedMarkdown table,
.text_cell_render table,
.rendered_html table {
    direction: rtl;
}
</style>
<div dir=rtl>


### تمرین سریع (۳ دقیقه)
یک بردار بسازید که ۳-۴ ویژگی عددی از پروژه‌ی خودتان (یا یک آیتم فرضی از آن) را نشان دهد. بعد یک ماتریس بسازید که همان ویژگی‌ها را برای ۵ آیتم مختلف نشان دهد.

In [6]:
# TODO: Feature vector of a single item from the project dataset (own_price, own_quality_level, own_seller_rating, price_gap_vs_cheapest_competitor)
# Data source: Dataset.xlsx -> design_matrix (Row 0)
my_item = torch.tensor([3.636802, 5.0, 94.597161, 3.448059])  # Completed item vector

# TODO: Matrix of 5 items with the same features
# Data source: Dataset.xlsx -> design_matrix (Rows 0 to 4)
my_dataset = torch.tensor([
    [3.636802, 5.0, 94.597161, 3.448059],
    [14.691189, 5.0, 94.597161, 12.193492],
    [4.829448, 5.0, 94.597161, 4.719306],
    [7.805090, 5.0, 94.597161, 1.508667],
    [1.124621, 5.0, 94.597161, -2.952796]
])  # Completed dataset matrix (5 items x 4 features)

print("shape یک آیتم:", my_item.shape)
print("shape کل دیتاست کوچک:", my_dataset.shape)

shape یک آیتم: torch.Size([4])
shape کل دیتاست کوچک: torch.Size([5, 4])


<style>
/* =====================================================================
   persian-rtl-jupyter.css
   هدف: نمایش درست متن فارسی (راست‌به‌چپ) همراه با فرمول‌های ریاضی
   (MathJax) و کد/متن انگلیسی (چپ‌به‌راست) در سلول‌های Markdown نوت‌بوک
   جوپیتر.

   این فایل برای سه محیط پوشش داده شده:
     1) JupyterLab (کلاس‌های .jp-*)
     2) Notebook کلاسیک (کلاس‌های .text_cell_render / .rendered_html)
     3) خروجی nbconvert به HTML و VS Code Jupyter (.rendered_html)
   ===================================================================== */

/* --- 1) جهت پایه‌ی سلول‌های مارک‌داون: راست‌به‌چپ ---
   معادل عملکرد dir="rtl": direction + unicode-bidi:isolate روی کانتینر،
   و direction:rtl روی *همه‌ی* فرزندان (strong, em, li, ...) تا هیچ تگی
   (مثل **بولد**) استایل را گم نکند. */
.jp-MarkdownCell .jp-RenderedMarkdown,
.text_cell_render,
.rendered_html {
    direction: rtl;
    text-align: right;
    unicode-bidi: isolate;
    font-family: "Vazirmatn", "Vazir", "Tahoma", "IRANSans", sans-serif;
    line-height: 1.9;
}

.jp-MarkdownCell .jp-RenderedMarkdown *,
.text_cell_render *,
.rendered_html * {
    direction: rtl;
    unicode-bidi: isolate;
}

/* --- 2) فرمول‌های ریاضی: همیشه چپ‌به‌راست و مستقل از متن اطراف --- */
.jp-MarkdownCell .jp-RenderedMarkdown mjx-container,
.text_cell_render mjx-container,
.rendered_html mjx-container,
.jp-MarkdownCell .jp-RenderedMarkdown .MathJax,
.text_cell_render .MathJax,
.rendered_html .MathJax,
.jp-MarkdownCell .jp-RenderedMarkdown .MathJax_Display,
.text_cell_render .MathJax_Display,
.rendered_html .MathJax_Display {
    direction: ltr;
    unicode-bidi: isolate;               /* فرمول یک "جزیره‌ی" مستقل LTR می‌شود */
    text-align: left;
}

/* فرمول‌های نمایشی (روی خط جدا، $$...$$) وسط‌چین بمانند */
mjx-container[display="true"] {
    text-align: center !important;
}

/* --- 3) کد inline و بلوک‌های کد: چپ‌به‌راست --- */
.jp-MarkdownCell .jp-RenderedMarkdown code,
.jp-MarkdownCell .jp-RenderedMarkdown pre,
.text_cell_render code,
.text_cell_render pre,
.rendered_html code,
.rendered_html pre {
    direction: ltr;
    unicode-bidi: isolate;
    text-align: left;
    font-family: "Fira Code", "Consolas", monospace;
}

/* --- 4) لیست‌ها و جدول‌ها هم راست‌چین شوند --- */
.jp-MarkdownCell .jp-RenderedMarkdown ul,
.jp-MarkdownCell .jp-RenderedMarkdown ol,
.text_cell_render ul,
.text_cell_render ol,
.rendered_html ul,
.rendered_html ol {
    padding-right: 1.5em;
    padding-left: 0;
}

.jp-MarkdownCell .jp-RenderedMarkdown table,
.text_cell_render table,
.rendered_html table {
    direction: rtl;
}
</style>
<div dir=rtl>


## بخش ۲: عملیات پایه — جمع، ضرب و عملیات عنصر‌به‌عنصر

📘 وقتی دو آرایه **دقیقاً هم‌شکل** باشند، جمع/تفریق/ضرب/تقسیم آن‌ها به‌صورت **عنصر‌به‌عنصر (element-wise)** انجام می‌شود — یعنی هر عنصر فقط با عنصر «هم‌جایگاه» خودش در آرایه‌ی دیگر ترکیب می‌شود.

In [7]:
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([10.0, 20.0, 30.0])

print("جمع عنصر‌به‌عنصر:", a + b)
print("ضرب عنصر‌به‌عنصر:", a * b)
print("توان عنصر‌به‌عنصر:", a ** 2)


جمع عنصر‌به‌عنصر: tensor([11., 22., 33.])
ضرب عنصر‌به‌عنصر: tensor([10., 40., 90.])
توان عنصر‌به‌عنصر: tensor([1., 4., 9.])


<style>
/* =====================================================================
   persian-rtl-jupyter.css
   هدف: نمایش درست متن فارسی (راست‌به‌چپ) همراه با فرمول‌های ریاضی
   (MathJax) و کد/متن انگلیسی (چپ‌به‌راست) در سلول‌های Markdown نوت‌بوک
   جوپیتر.

   این فایل برای سه محیط پوشش داده شده:
     1) JupyterLab (کلاس‌های .jp-*)
     2) Notebook کلاسیک (کلاس‌های .text_cell_render / .rendered_html)
     3) خروجی nbconvert به HTML و VS Code Jupyter (.rendered_html)
   ===================================================================== */

/* --- 1) جهت پایه‌ی سلول‌های مارک‌داون: راست‌به‌چپ ---
   معادل عملکرد dir="rtl": direction + unicode-bidi:isolate روی کانتینر،
   و direction:rtl روی *همه‌ی* فرزندان (strong, em, li, ...) تا هیچ تگی
   (مثل **بولد**) استایل را گم نکند. */
.jp-MarkdownCell .jp-RenderedMarkdown,
.text_cell_render,
.rendered_html {
    direction: rtl;
    text-align: right;
    unicode-bidi: isolate;
    font-family: "Vazirmatn", "Vazir", "Tahoma", "IRANSans", sans-serif;
    line-height: 1.9;
}

.jp-MarkdownCell .jp-RenderedMarkdown *,
.text_cell_render *,
.rendered_html * {
    direction: rtl;
    unicode-bidi: isolate;
}

/* --- 2) فرمول‌های ریاضی: همیشه چپ‌به‌راست و مستقل از متن اطراف --- */
.jp-MarkdownCell .jp-RenderedMarkdown mjx-container,
.text_cell_render mjx-container,
.rendered_html mjx-container,
.jp-MarkdownCell .jp-RenderedMarkdown .MathJax,
.text_cell_render .MathJax,
.rendered_html .MathJax,
.jp-MarkdownCell .jp-RenderedMarkdown .MathJax_Display,
.text_cell_render .MathJax_Display,
.rendered_html .MathJax_Display {
    direction: ltr;
    unicode-bidi: isolate;               /* فرمول یک "جزیره‌ی" مستقل LTR می‌شود */
    text-align: left;
}

/* فرمول‌های نمایشی (روی خط جدا، $$...$$) وسط‌چین بمانند */
mjx-container[display="true"] {
    text-align: center !important;
}

/* --- 3) کد inline و بلوک‌های کد: چپ‌به‌راست --- */
.jp-MarkdownCell .jp-RenderedMarkdown code,
.jp-MarkdownCell .jp-RenderedMarkdown pre,
.text_cell_render code,
.text_cell_render pre,
.rendered_html code,
.rendered_html pre {
    direction: ltr;
    unicode-bidi: isolate;
    text-align: left;
    font-family: "Fira Code", "Consolas", monospace;
}

/* --- 4) لیست‌ها و جدول‌ها هم راست‌چین شوند --- */
.jp-MarkdownCell .jp-RenderedMarkdown ul,
.jp-MarkdownCell .jp-RenderedMarkdown ol,
.text_cell_render ul,
.text_cell_render ol,
.rendered_html ul,
.rendered_html ol {
    padding-right: 1.5em;
    padding-left: 0;
}

.jp-MarkdownCell .jp-RenderedMarkdown table,
.text_cell_render table,
.rendered_html table {
    direction: rtl;
}
</style>
<div dir=rtl>


### 🗣️ بحث دو‌نفره — ۱ دقیقه
اگر بخواهید دو بردار با طول‌های متفاوت (مثلاً یکی با ۳ عنصر و دیگری با ۵ عنصر) را جمع بزنید، فکر می‌کنید چه اتفاقی می‌افتد؟ حدستان را با نفر کناری بگویید، بعد در بخش بعد جواب دقیق را می‌بینیم.

<style>
/* =====================================================================
   persian-rtl-jupyter.css
   هدف: نمایش درست متن فارسی (راست‌به‌چپ) همراه با فرمول‌های ریاضی
   (MathJax) و کد/متن انگلیسی (چپ‌به‌راست) در سلول‌های Markdown نوت‌بوک
   جوپیتر.

   این فایل برای سه محیط پوشش داده شده:
     1) JupyterLab (کلاس‌های .jp-*)
     2) Notebook کلاسیک (کلاس‌های .text_cell_render / .rendered_html)
     3) خروجی nbconvert به HTML و VS Code Jupyter (.rendered_html)
   ===================================================================== */

/* --- 1) جهت پایه‌ی سلول‌های مارک‌داون: راست‌به‌چپ ---
   معادل عملکرد dir="rtl": direction + unicode-bidi:isolate روی کانتینر،
   و direction:rtl روی *همه‌ی* فرزندان (strong, em, li, ...) تا هیچ تگی
   (مثل **بولد**) استایل را گم نکند. */
.jp-MarkdownCell .jp-RenderedMarkdown,
.text_cell_render,
.rendered_html {
    direction: rtl;
    text-align: right;
    unicode-bidi: isolate;
    font-family: "Vazirmatn", "Vazir", "Tahoma", "IRANSans", sans-serif;
    line-height: 1.9;
}

.jp-MarkdownCell .jp-RenderedMarkdown *,
.text_cell_render *,
.rendered_html * {
    direction: rtl;
    unicode-bidi: isolate;
}

/* --- 2) فرمول‌های ریاضی: همیشه چپ‌به‌راست و مستقل از متن اطراف --- */
.jp-MarkdownCell .jp-RenderedMarkdown mjx-container,
.text_cell_render mjx-container,
.rendered_html mjx-container,
.jp-MarkdownCell .jp-RenderedMarkdown .MathJax,
.text_cell_render .MathJax,
.rendered_html .MathJax,
.jp-MarkdownCell .jp-RenderedMarkdown .MathJax_Display,
.text_cell_render .MathJax_Display,
.rendered_html .MathJax_Display {
    direction: ltr;
    unicode-bidi: isolate;               /* فرمول یک "جزیره‌ی" مستقل LTR می‌شود */
    text-align: left;
}

/* فرمول‌های نمایشی (روی خط جدا، $$...$$) وسط‌چین بمانند */
mjx-container[display="true"] {
    text-align: center !important;
}

/* --- 3) کد inline و بلوک‌های کد: چپ‌به‌راست --- */
.jp-MarkdownCell .jp-RenderedMarkdown code,
.jp-MarkdownCell .jp-RenderedMarkdown pre,
.text_cell_render code,
.text_cell_render pre,
.rendered_html code,
.rendered_html pre {
    direction: ltr;
    unicode-bidi: isolate;
    text-align: left;
    font-family: "Fira Code", "Consolas", monospace;
}

/* --- 4) لیست‌ها و جدول‌ها هم راست‌چین شوند --- */
.jp-MarkdownCell .jp-RenderedMarkdown ul,
.jp-MarkdownCell .jp-RenderedMarkdown ol,
.text_cell_render ul,
.text_cell_render ol,
.rendered_html ul,
.rendered_html ol {
    padding-right: 1.5em;
    padding-left: 0;
}

.jp-MarkdownCell .jp-RenderedMarkdown table,
.text_cell_render table,
.rendered_html table {
    direction: rtl;
}
</style>
<div dir=rtl>


## بخش ۳: Broadcasting — وقتی شکل‌ها یکسان نیستند

📘 اما در دنیای واقعی، خیلی وقت‌ها می‌خواهیم یک آرایه‌ی **بزرگ** را با یک آرایه‌ی **کوچک‌تر** ترکیب کنیم — مثلاً همان مثال قیمت‌ها: یک آرایه‌ی ۱ میلیون‌تایی را در یک عدد تنها (اسکالر) ضرب کردیم. **Broadcasting** قانونی است که به NumPy/PyTorch اجازه می‌دهد آرایه‌ی کوچک‌تر را به‌طور خودکار «تکرار مجازی» کند تا با آرایه‌ی بزرگ‌تر هم‌شکل شود — بدون اینکه واقعاً در حافظه کپی شود (پس هم سریع است، هم کم‌حافظه).

**قانون Broadcasting (خلاصه):** برای مقایسه‌ی دو شکل، از سمت راست شروع می‌کنیم؛ در هر بُعد، اندازه‌ها باید یا **برابر** باشند یا یکی از آن‌ها **۱** باشد. اگر این شرط برقرار نباشد، خطا می‌گیرید.

بیایید چند سطح پیچیدگی را با هم ببینیم.

In [8]:
# سطح ۱: اسکالر + بردار (همان کاری که با قیمت‌ها کردیم)
prices_sample = torch.tensor([100.0, 200.0, 300.0])
print("بردار + اسکالر:", prices_sample + 10)
print("→ shape بردار:", tuple(prices_sample.shape), " | shape اسکالر: () → broadcast به (3,)")


بردار + اسکالر: tensor([110., 210., 310.])
→ shape بردار: (3,)  | shape اسکالر: () → broadcast به (3,)


In [9]:
# سطح ۲: ماتریس + بردار سطری (رایج‌ترین حالت در یادگیری ماشین)
# فرض کنید هر سطر یک کاربر و هر ستون یک ویژگی است؛ می‌خواهیم به هر ویژگی یک offset متفاوت اضافه کنیم
user_features = torch.tensor([[1.0, 2.0, 3.0],
                                [4.0, 5.0, 6.0],
                                [7.0, 8.0, 9.0]])  # shape: (3 کاربر, 3 ویژگی)

feature_offset = torch.tensor([10.0, 20.0, 30.0])  # shape: (3 ویژگی,)

result = user_features + feature_offset
print("ماتریس (3,3) + بردار (3,):\n", result)
print("\n📌 بردار offset به‌طور خودکار روی هر ۳ سطر 'تکرار' شد — بدون این‌که ما حلقه بزنیم یا کپی صریح بسازیم.")


ماتریس (3,3) + بردار (3,):
 tensor([[11., 22., 33.],
        [14., 25., 36.],
        [17., 28., 39.]])

📌 بردار offset به‌طور خودکار روی هر ۳ سطر 'تکرار' شد — بدون این‌که ما حلقه بزنیم یا کپی صریح بسازیم.


In [10]:
# سطح ۳: بردار ستونی + بردار سطری -> یک ماتریس کامل (outer broadcasting)
col = torch.tensor([[1.0], [2.0], [3.0]])   # shape: (3, 1)
row = torch.tensor([10.0, 20.0, 30.0])       # shape: (3,)  → در broadcasting مثل (1, 3) دیده می‌شود

print("shape بردار ستونی:", tuple(col.shape))
print("shape بردار سطری:", tuple(row.shape))
print("\nحاصل col + row (یک ماتریس کامل ۳×۳ از دل دو بردار):\n", col + row)


shape بردار ستونی: (3, 1)
shape بردار سطری: (3,)

حاصل col + row (یک ماتریس کامل ۳×۳ از دل دو بردار):
 tensor([[11., 21., 31.],
        [12., 22., 32.],
        [13., 23., 33.]])


<style>
/* =====================================================================
   persian-rtl-jupyter.css
   هدف: نمایش درست متن فارسی (راست‌به‌چپ) همراه با فرمول‌های ریاضی
   (MathJax) و کد/متن انگلیسی (چپ‌به‌راست) در سلول‌های Markdown نوت‌بوک
   جوپیتر.

   این فایل برای سه محیط پوشش داده شده:
     1) JupyterLab (کلاس‌های .jp-*)
     2) Notebook کلاسیک (کلاس‌های .text_cell_render / .rendered_html)
     3) خروجی nbconvert به HTML و VS Code Jupyter (.rendered_html)
   ===================================================================== */

/* --- 1) جهت پایه‌ی سلول‌های مارک‌داون: راست‌به‌چپ ---
   معادل عملکرد dir="rtl": direction + unicode-bidi:isolate روی کانتینر،
   و direction:rtl روی *همه‌ی* فرزندان (strong, em, li, ...) تا هیچ تگی
   (مثل **بولد**) استایل را گم نکند. */
.jp-MarkdownCell .jp-RenderedMarkdown,
.text_cell_render,
.rendered_html {
    direction: rtl;
    text-align: right;
    unicode-bidi: isolate;
    font-family: "Vazirmatn", "Vazir", "Tahoma", "IRANSans", sans-serif;
    line-height: 1.9;
}

.jp-MarkdownCell .jp-RenderedMarkdown *,
.text_cell_render *,
.rendered_html * {
    direction: rtl;
    unicode-bidi: isolate;
}

/* --- 2) فرمول‌های ریاضی: همیشه چپ‌به‌راست و مستقل از متن اطراف --- */
.jp-MarkdownCell .jp-RenderedMarkdown mjx-container,
.text_cell_render mjx-container,
.rendered_html mjx-container,
.jp-MarkdownCell .jp-RenderedMarkdown .MathJax,
.text_cell_render .MathJax,
.rendered_html .MathJax,
.jp-MarkdownCell .jp-RenderedMarkdown .MathJax_Display,
.text_cell_render .MathJax_Display,
.rendered_html .MathJax_Display {
    direction: ltr;
    unicode-bidi: isolate;               /* فرمول یک "جزیره‌ی" مستقل LTR می‌شود */
    text-align: left;
}

/* فرمول‌های نمایشی (روی خط جدا، $$...$$) وسط‌چین بمانند */
mjx-container[display="true"] {
    text-align: center !important;
}

/* --- 3) کد inline و بلوک‌های کد: چپ‌به‌راست --- */
.jp-MarkdownCell .jp-RenderedMarkdown code,
.jp-MarkdownCell .jp-RenderedMarkdown pre,
.text_cell_render code,
.text_cell_render pre,
.rendered_html code,
.rendered_html pre {
    direction: ltr;
    unicode-bidi: isolate;
    text-align: left;
    font-family: "Fira Code", "Consolas", monospace;
}

/* --- 4) لیست‌ها و جدول‌ها هم راست‌چین شوند --- */
.jp-MarkdownCell .jp-RenderedMarkdown ul,
.jp-MarkdownCell .jp-RenderedMarkdown ol,
.text_cell_render ul,
.text_cell_render ol,
.rendered_html ul,
.rendered_html ol {
    padding-right: 1.5em;
    padding-left: 0;
}

.jp-MarkdownCell .jp-RenderedMarkdown table,
.text_cell_render table,
.rendered_html table {
    direction: rtl;
}
</style>
<div dir=rtl>


### 🗣️ بحث دو‌نفره (Think–Pair–Share) — ۲ دقیقه
به نتیجه‌ی سلول بالا نگاه کنید. با نفر کناری‌تان توضیح دهید: چرا جمع یک بردار ستونی (۳×۱) با یک بردار سطری (۳،) یک ماتریس کامل ۳×۳ تولید کرد، نه یک بردار؟

<style>
/* =====================================================================
   persian-rtl-jupyter.css
   هدف: نمایش درست متن فارسی (راست‌به‌چپ) همراه با فرمول‌های ریاضی
   (MathJax) و کد/متن انگلیسی (چپ‌به‌راست) در سلول‌های Markdown نوت‌بوک
   جوپیتر.

   این فایل برای سه محیط پوشش داده شده:
     1) JupyterLab (کلاس‌های .jp-*)
     2) Notebook کلاسیک (کلاس‌های .text_cell_render / .rendered_html)
     3) خروجی nbconvert به HTML و VS Code Jupyter (.rendered_html)
   ===================================================================== */

/* --- 1) جهت پایه‌ی سلول‌های مارک‌داون: راست‌به‌چپ ---
   معادل عملکرد dir="rtl": direction + unicode-bidi:isolate روی کانتینر،
   و direction:rtl روی *همه‌ی* فرزندان (strong, em, li, ...) تا هیچ تگی
   (مثل **بولد**) استایل را گم نکند. */
.jp-MarkdownCell .jp-RenderedMarkdown,
.text_cell_render,
.rendered_html {
    direction: rtl;
    text-align: right;
    unicode-bidi: isolate;
    font-family: "Vazirmatn", "Vazir", "Tahoma", "IRANSans", sans-serif;
    line-height: 1.9;
}

.jp-MarkdownCell .jp-RenderedMarkdown *,
.text_cell_render *,
.rendered_html * {
    direction: rtl;
    unicode-bidi: isolate;
}

/* --- 2) فرمول‌های ریاضی: همیشه چپ‌به‌راست و مستقل از متن اطراف --- */
.jp-MarkdownCell .jp-RenderedMarkdown mjx-container,
.text_cell_render mjx-container,
.rendered_html mjx-container,
.jp-MarkdownCell .jp-RenderedMarkdown .MathJax,
.text_cell_render .MathJax,
.rendered_html .MathJax,
.jp-MarkdownCell .jp-RenderedMarkdown .MathJax_Display,
.text_cell_render .MathJax_Display,
.rendered_html .MathJax_Display {
    direction: ltr;
    unicode-bidi: isolate;               /* فرمول یک "جزیره‌ی" مستقل LTR می‌شود */
    text-align: left;
}

/* فرمول‌های نمایشی (روی خط جدا، $$...$$) وسط‌چین بمانند */
mjx-container[display="true"] {
    text-align: center !important;
}

/* --- 3) کد inline و بلوک‌های کد: چپ‌به‌راست --- */
.jp-MarkdownCell .jp-RenderedMarkdown code,
.jp-MarkdownCell .jp-RenderedMarkdown pre,
.text_cell_render code,
.text_cell_render pre,
.rendered_html code,
.rendered_html pre {
    direction: ltr;
    unicode-bidi: isolate;
    text-align: left;
    font-family: "Fira Code", "Consolas", monospace;
}

/* --- 4) لیست‌ها و جدول‌ها هم راست‌چین شوند --- */
.jp-MarkdownCell .jp-RenderedMarkdown ul,
.jp-MarkdownCell .jp-RenderedMarkdown ol,
.text_cell_render ul,
.text_cell_render ol,
.rendered_html ul,
.rendered_html ol {
    padding-right: 1.5em;
    padding-left: 0;
}

.jp-MarkdownCell .jp-RenderedMarkdown table,
.text_cell_render table,
.rendered_html table {
    direction: rtl;
}
</style>
<div dir=rtl>


# قاعده‌ی Broadcasting در NumPy و PyTorch

##  تعریف رسمی

فرض کنید دو آرایه یا Tensor با شکل‌های

$$
(n_1,n_2,\ldots,n_r)
$$

و

$$
(m_1,m_2,\ldots,m_s)
$$

داریم.

برای بررسی امکان Broadcasting، شکل دو آرایه را از **آخرین بُعد به سمت اولین بُعد** با یکدیگر مقایسه می‌کنیم.

اگر تعداد ابعاد دو آرایه برابر نباشد، شکل آرایه‌ای که ابعاد کمتری دارد، به‌صورت مفهومی از سمت چپ با $1$ تکمیل می‌شود.

دو بُعد متناظر با یکدیگر **سازگار** هستند اگر و تنها اگر یکی از شرایط زیر برقرار باشد:

$$
n_i=m_i
$$

یا

$$
n_i=1
$$

یا

$$
m_i=1.
$$

بنابراین قاعده‌ی اصلی Broadcasting عبارت است از:


\boxed{
\text{دو بُعد در صورتی سازگارند که برابر باشند یا یکی از آن‌ها برابر }1\text{ باشد.}
}


اگر این شرط برای تمام ابعاد متناظر برقرار باشد، Broadcasting امکان‌پذیر است.

---

## ۲. شکل Tensor حاصل

اگر Broadcasting امکان‌پذیر باشد، شکل حاصل در هر بُعد برابر با بزرگ‌ترین مقدار آن دو بُعد است:

$$
\boxed{
\max(n_i,m_i)
}
$$

بنابراین اگر دو Tensor دارای شکل‌های

$$
(2,3)
$$

و

$$
(1,3)
$$

باشند، شکل نتیجه برابر است با:

$$
(2,3).
$$

---

# ۳. چرا از سمت راست مقایسه می‌کنیم؟

فرض کنید:

$$
A\in\mathbb{R}^{2\times3}
$$

و

$$
b\in\mathbb{R}^{3}.
$$

شکل‌ها عبارت‌اند از:

$$
(2,3)
$$

و

$$
(3).
$$

چون Tensor دوم فقط یک بُعد دارد، برای مقایسه‌ی ابعاد، آن را به‌صورت مفهومی با یک $1$ در سمت چپ در نظر می‌گیریم:

$$
(2,3)
$$

و

$$
(1,3).
$$

حالا از راست مقایسه می‌کنیم:

$$
\begin{array}{cc}
2 & 3\\
1 & 3
\end{array}
$$

در بُعد آخر:

$$
3=3
$$

و در بُعد اول:

$$
2\neq1
$$

اما یکی از آن‌ها $1$ است؛ بنابراین این دو بُعد نیز سازگارند.

پس:

$$
(2,3)+(3)\longrightarrow(2,3).
$$

---

# ۴. مثال اول: عدد و بردار

فرض کنید:

```python
import numpy as np

x = np.array([1, 2, 3])
y = 10

x + y
```

نتیجه:

```python
array([11, 12, 13])
```

از نظر شکل:

$$
x:\quad(3)
$$

و

$$
y:\quad()
$$

عدد اسکالر هیچ بُعدی ندارد.

در Broadcasting، اسکالر می‌تواند با هر شکلی سازگار شود.

به‌صورت مفهومی:

$$
\begin{bmatrix}
1&2&3
\end{bmatrix}
+
10
$$

مانند این است که بنویسیم:

$$
\begin{bmatrix}
1&2&3
\end{bmatrix}
+
\begin{bmatrix}
10&10&10
\end{bmatrix}
$$

و در نتیجه:

$$
\begin{bmatrix}
11&12&13
\end{bmatrix}.
$$

البته این تکرار صرفاً برای فهم Broadcasting است و الزاماً به معنای ایجاد یک آرایه‌ی جدید از سه مقدار $10$ در حافظه نیست.

همین قاعده در PyTorch نیز وجود دارد:

```python
import torch

x = torch.tensor([1., 2., 3.])
y = 10

x + y
```

نتیجه:

```python
tensor([11., 12., 13.])
```

---

# ۵. مثال دوم: ماتریس و بردار

فرض کنید:

$$
A=
\begin{bmatrix}
1&2&3\\
4&5&6
\end{bmatrix}
$$

و

$$
b=
\begin{bmatrix}
10&20&30
\end{bmatrix}.
$$

داریم:

$$
\operatorname{shape}(A)=(2,3)
$$

و

$$
\operatorname{shape}(b)=(3).
$$

برای بررسی Broadcasting:

$$
(2,3)
$$

و

$$
(3)
$$

را از سمت راست مقایسه می‌کنیم:

$$
\begin{array}{cc}
2&3\\
1&3
\end{array}
$$

بنابراین سازگارند.

شکل حاصل:

$$
(2,3).
$$

از نظر مفهومی، بردار $b$ برای هر سطر تکرار می‌شود:

$$
\begin{bmatrix}
1&2&3\\
4&5&6
\end{bmatrix}
+
\begin{bmatrix}
10&20&30\\
10&20&30
\end{bmatrix}
$$

و نتیجه:

$$
\begin{bmatrix}
11&22&33\\
14&25&36
\end{bmatrix}.
$$

در NumPy:

```python
A = np.array([
    [1, 2, 3],
    [4, 5, 6]
])

b = np.array([10, 20, 30])

A + b
```

و در PyTorch:

```python
A = torch.tensor([
    [1., 2., 3.],
    [4., 5., 6.]
])

b = torch.tensor([10., 20., 30.])

A + b
```

در هر دو کتابخانه نتیجه یکسان است.

---

# ۶. مثال سوم: Broadcasting در راستای ستون‌ها

حالا به جای بردار سطری، Tensor با شکل

$$
(2,1)
$$

داریم:

$$
b=
\begin{bmatrix}
10\\
20
\end{bmatrix}.
$$

و:

$$
A\in\mathbb{R}^{2\times3}.
$$

بنابراین:

$$
\operatorname{shape}(A)=(2,3)
$$

و

$$
\operatorname{shape}(b)=(2,1).
$$

مقایسه:

$$
\begin{array}{cc}
2&3\\
2&1
\end{array}
$$

در بُعد اول:

$$
2=2
$$

و در بُعد دوم:

$$
3\neq1
$$

اما یکی از آن‌ها $1$ است.

پس Broadcasting امکان‌پذیر است:

$$
(2,3)+(2,1)\longrightarrow(2,3).
$$

از نظر مفهومی:

$$
\begin{bmatrix}
1&2&3\\
4&5&6
\end{bmatrix}
+
\begin{bmatrix}
10&10&10\\
20&20&20
\end{bmatrix}
$$

و بنابراین:

$$
\begin{bmatrix}
11&12&13\\
24&25&26
\end{bmatrix}.
$$

نکته‌ی مهم این است که در مثال قبلی، بردار در راستای سطرها Broadcasting شد، ولی در اینجا در راستای ستون‌ها.

---

# ۷. مثالی که Broadcasting امکان‌پذیر نیست

فرض کنید:

$$
A\in\mathbb{R}^{2\times3}
$$

و

$$
b\in\mathbb{R}^{2}.
$$

شکل‌ها:

$$
(2,3)
$$

و

$$
(2).
$$

برای مقایسه، شکل دوم را به‌صورت مفهومی به شکل

$$
(1,2)
$$

در نظر می‌گیریم:

$$
\begin{array}{cc}
2&3\\
1&2
\end{array}
$$

در بُعد آخر داریم:

$$
3\neq2
$$

و هیچ‌کدام $1$ نیستند.

بنابراین:

$$
\boxed{
(2,3)+(2)
\quad\text{امکان‌پذیر نیست.}
}
$$

در NumPy معمولاً خطایی مانند زیر دریافت می‌کنیم:

```text
ValueError: operands could not be broadcast together
```

و PyTorch نیز خطای مربوط به ناسازگاری ابعاد ایجاد می‌کند.

---

# ۸. مثال مهم: تفاوت $(3)$، $(1,3)$ و $(3,1)$

این سه شکل را نباید یکسان در نظر گرفت:

$$
(3)
\qquad
(1,3)
\qquad
(3,1).
$$

مثلاً:

$$
(3)
$$

یک Tensor یک‌بعدی است.

اما:

$$
(1,3)
$$

یک Tensor دوبعدی با یک سطر و سه ستون است.

و:

$$
(3,1)
$$

یک Tensor دوبعدی با سه سطر و یک ستون است.

برای مثال:

$$
(2,3)+(1,3)\longrightarrow(2,3)
$$

اما:

$$
(2,3)+(3,1)
$$

قابل Broadcasting نیست.

دلیل آن را می‌توان با مقایسه‌ی ابعاد دید:

$$
\begin{array}{cc}
2&3\\
3&1
\end{array}
$$

در بُعد آخر:

$$
3\neq1
$$

مشکلی وجود ندارد، اما در بُعد اول:

$$
2\neq3
$$

و هیچ‌کدام $1$ نیستند.

---

# ۹. یک مثال مهم در Machine Learning

فرض کنید خروجی یک لایه‌ی شبکه‌ی عصبی برای $4$ نمونه و $3$ ویژگی را داشته باشیم:

$$
X\in\mathbb{R}^{4\times3}.
$$

مثلاً:

$$
X=
\begin{bmatrix}
x_{11}&x_{12}&x_{13}\\
x_{21}&x_{22}&x_{23}\\
x_{31}&x_{32}&x_{33}\\
x_{41}&x_{42}&x_{43}
\end{bmatrix}.
$$

حالا یک بردار bias داریم:

$$
b=
\begin{bmatrix}
b_1&b_2&b_3
\end{bmatrix}
\in\mathbb{R}^{3}.
$$

در شبکه‌ی عصبی می‌توانیم بنویسیم:

$$
Y=X+b.
$$

Broadcasting باعث می‌شود:

$$
Y=
\begin{bmatrix}
x_{11}+b_1&x_{12}+b_2&x_{13}+b_3\\
x_{21}+b_1&x_{22}+b_2&x_{23}+b_3\\
x_{31}+b_1&x_{32}+b_2&x_{33}+b_3\\
x_{41}+b_1&x_{42}+b_2&x_{43}+b_3
\end{bmatrix}.
$$

یعنی یک بردار $3$-بعدی می‌تواند به‌طور هم‌زمان به هر چهار نمونه اعمال شود.

این یکی از کاربردهای بسیار مهم Broadcasting در شبکه‌های عصبی است.

---

# ۱۰. خلاصه‌ی قاعده

برای بررسی Broadcasting دو Tensor:

1. شکل آن‌ها را از **آخرین بُعد به سمت اولین بُعد** مقایسه می‌کنیم.

2. اگر تعداد ابعاد برابر نباشد، شکل Tensor با ابعاد کمتر را از سمت چپ با $1$ کامل می‌کنیم.

3. هر دو بُعد متناظر باید یکی از این شرایط را داشته باشند:

$$
\boxed{
n=m
\quad\text{یا}\quad
n=1
\quad\text{یا}\quad
m=1
}
$$

4. اگر این شرط برای تمام ابعاد برقرار باشد، Broadcasting امکان‌پذیر است.

5. شکل نتیجه در هر بُعد برابر با بزرگ‌ترین اندازه‌ی آن بُعد است.

بنابراین می‌توان قاعده را به‌صورت فشرده چنین بیان کرد:

$$
\boxed{
\text{Broadcasting}
=
\text{مقایسه‌ی ابعاد از راست به چپ}
+
\text{برابری یا وجود }1
}
$$

این قاعده در **NumPy** و **PyTorch** یکسان است و اساس بسیاری از محاسبات برداری و Tensorی در یادگیری ماشین را تشکیل می‌دهد.


<style>
/* =====================================================================
   persian-rtl-jupyter.css
   هدف: نمایش درست متن فارسی (راست‌به‌چپ) همراه با فرمول‌های ریاضی
   (MathJax) و کد/متن انگلیسی (چپ‌به‌راست) در سلول‌های Markdown نوت‌بوک
   جوپیتر.

   این فایل برای سه محیط پوشش داده شده:
     1) JupyterLab (کلاس‌های .jp-*)
     2) Notebook کلاسیک (کلاس‌های .text_cell_render / .rendered_html)
     3) خروجی nbconvert به HTML و VS Code Jupyter (.rendered_html)
   ===================================================================== */

/* --- 1) جهت پایه‌ی سلول‌های مارک‌داون: راست‌به‌چپ ---
   معادل عملکرد dir="rtl": direction + unicode-bidi:isolate روی کانتینر،
   و direction:rtl روی *همه‌ی* فرزندان (strong, em, li, ...) تا هیچ تگی
   (مثل **بولد**) استایل را گم نکند. */
.jp-MarkdownCell .jp-RenderedMarkdown,
.text_cell_render,
.rendered_html {
    direction: rtl;
    text-align: right;
    unicode-bidi: isolate;
    font-family: "Vazirmatn", "Vazir", "Tahoma", "IRANSans", sans-serif;
    line-height: 1.9;
}

.jp-MarkdownCell .jp-RenderedMarkdown *,
.text_cell_render *,
.rendered_html * {
    direction: rtl;
    unicode-bidi: isolate;
}

/* --- 2) فرمول‌های ریاضی: همیشه چپ‌به‌راست و مستقل از متن اطراف --- */
.jp-MarkdownCell .jp-RenderedMarkdown mjx-container,
.text_cell_render mjx-container,
.rendered_html mjx-container,
.jp-MarkdownCell .jp-RenderedMarkdown .MathJax,
.text_cell_render .MathJax,
.rendered_html .MathJax,
.jp-MarkdownCell .jp-RenderedMarkdown .MathJax_Display,
.text_cell_render .MathJax_Display,
.rendered_html .MathJax_Display {
    direction: ltr;
    unicode-bidi: isolate;               /* فرمول یک "جزیره‌ی" مستقل LTR می‌شود */
    text-align: left;
}

/* فرمول‌های نمایشی (روی خط جدا، $$...$$) وسط‌چین بمانند */
mjx-container[display="true"] {
    text-align: center !important;
}

/* --- 3) کد inline و بلوک‌های کد: چپ‌به‌راست --- */
.jp-MarkdownCell .jp-RenderedMarkdown code,
.jp-MarkdownCell .jp-RenderedMarkdown pre,
.text_cell_render code,
.text_cell_render pre,
.rendered_html code,
.rendered_html pre {
    direction: ltr;
    unicode-bidi: isolate;
    text-align: left;
    font-family: "Fira Code", "Consolas", monospace;
}

/* --- 4) لیست‌ها و جدول‌ها هم راست‌چین شوند --- */
.jp-MarkdownCell .jp-RenderedMarkdown ul,
.jp-MarkdownCell .jp-RenderedMarkdown ol,
.text_cell_render ul,
.text_cell_render ol,
.rendered_html ul,
.rendered_html ol {
    padding-right: 1.5em;
    padding-left: 0;
}

.jp-MarkdownCell .jp-RenderedMarkdown table,
.text_cell_render table,
.rendered_html table {
    direction: rtl;
}
</style>
<div dir=rtl>


## بخش ۴: یک کاربرد واقعی — نرمال‌سازی داده با Broadcasting

📘 یکی از رایج‌ترین کاربردهای Broadcasting در یادگیری ماشین، **نرمال‌سازی (Normalization / Standardization)** است: هر ویژگی (ستون) را طوری تغییر می‌دهیم که میانگینش ۰ و انحراف‌معیارش ۱ شود. این کار باعث می‌شود ویژگی‌هایی با مقیاس‌های خیلی متفاوت (مثلاً «سن» بین ۱۸ تا ۶۰، و «درآمد» بین میلیون‌ها تومان) به‌طور منصفانه در مدل شرکت کنند. (این مفهوم را کامل‌تر در بخش آمار درس می‌بینیم؛ امروز فقط از زاویه‌ی Broadcasting به آن نگاه می‌کنیم.)

فرمول: $\text{نرمال‌شده} = \dfrac{\text{داده} - \text{میانگین هر ستون}}{\text{انحراف‌معیار هر ستون}}$

In [11]:
# داده‌ی فرضی: ۴ کاربر، ۲ ویژگی با مقیاس خیلی متفاوت (سن و درآمد)
data = torch.tensor([[18.0, 3_000_000.0],
                      [25.0, 5_000_000.0],
                      [40.0, 12_000_000.0],
                      [60.0, 25_000_000.0]])

col_mean = data.mean(dim=0)   # میانگین هر ستون؛ shape: (2,)
col_std = data.std(dim=0)     # انحراف‌معیار هر ستون؛ shape: (2,)

print("shape داده:", tuple(data.shape))
print("میانگین هر ستون:", col_mean.tolist())
print("انحراف‌معیار هر ستون:", col_std.tolist())

normalized = (data - col_mean) / col_std   # Broadcasting: (4,2) با (2,) ترکیب می‌شود
print("\nداده‌ی نرمال‌شده:\n", normalized)
print("\nمیانگین جدید هر ستون (باید نزدیک صفر باشد):", normalized.mean(dim=0).tolist())


shape داده: (4, 2)
میانگین هر ستون: [35.75, 11250000.0]
انحراف‌معیار هر ستون: [18.589872360229492, 9945686.0]

داده‌ی نرمال‌شده:
 tensor([[-0.9548, -0.8295],
        [-0.5783, -0.6284],
        [ 0.2286,  0.0754],
        [ 1.3045,  1.3825]])

میانگین جدید هر ستون (باید نزدیک صفر باشد): [0.0, 0.0]


<style>
/* =====================================================================
   persian-rtl-jupyter.css
   هدف: نمایش درست متن فارسی (راست‌به‌چپ) همراه با فرمول‌های ریاضی
   (MathJax) و کد/متن انگلیسی (چپ‌به‌راست) در سلول‌های Markdown نوت‌بوک
   جوپیتر.

   این فایل برای سه محیط پوشش داده شده:
     1) JupyterLab (کلاس‌های .jp-*)
     2) Notebook کلاسیک (کلاس‌های .text_cell_render / .rendered_html)
     3) خروجی nbconvert به HTML و VS Code Jupyter (.rendered_html)
   ===================================================================== */

/* --- 1) جهت پایه‌ی سلول‌های مارک‌داون: راست‌به‌چپ ---
   معادل عملکرد dir="rtl": direction + unicode-bidi:isolate روی کانتینر،
   و direction:rtl روی *همه‌ی* فرزندان (strong, em, li, ...) تا هیچ تگی
   (مثل **بولد**) استایل را گم نکند. */
.jp-MarkdownCell .jp-RenderedMarkdown,
.text_cell_render,
.rendered_html {
    direction: rtl;
    text-align: right;
    unicode-bidi: isolate;
    font-family: "Vazirmatn", "Vazir", "Tahoma", "IRANSans", sans-serif;
    line-height: 1.9;
}

.jp-MarkdownCell .jp-RenderedMarkdown *,
.text_cell_render *,
.rendered_html * {
    direction: rtl;
    unicode-bidi: isolate;
}

/* --- 2) فرمول‌های ریاضی: همیشه چپ‌به‌راست و مستقل از متن اطراف --- */
.jp-MarkdownCell .jp-RenderedMarkdown mjx-container,
.text_cell_render mjx-container,
.rendered_html mjx-container,
.jp-MarkdownCell .jp-RenderedMarkdown .MathJax,
.text_cell_render .MathJax,
.rendered_html .MathJax,
.jp-MarkdownCell .jp-RenderedMarkdown .MathJax_Display,
.text_cell_render .MathJax_Display,
.rendered_html .MathJax_Display {
    direction: ltr;
    unicode-bidi: isolate;               /* فرمول یک "جزیره‌ی" مستقل LTR می‌شود */
    text-align: left;
}

/* فرمول‌های نمایشی (روی خط جدا، $$...$$) وسط‌چین بمانند */
mjx-container[display="true"] {
    text-align: center !important;
}

/* --- 3) کد inline و بلوک‌های کد: چپ‌به‌راست --- */
.jp-MarkdownCell .jp-RenderedMarkdown code,
.jp-MarkdownCell .jp-RenderedMarkdown pre,
.text_cell_render code,
.text_cell_render pre,
.rendered_html code,
.rendered_html pre {
    direction: ltr;
    unicode-bidi: isolate;
    text-align: left;
    font-family: "Fira Code", "Consolas", monospace;
}

/* --- 4) لیست‌ها و جدول‌ها هم راست‌چین شوند --- */
.jp-MarkdownCell .jp-RenderedMarkdown ul,
.jp-MarkdownCell .jp-RenderedMarkdown ol,
.text_cell_render ul,
.text_cell_render ol,
.rendered_html ul,
.rendered_html ol {
    padding-right: 1.5em;
    padding-left: 0;
}

.jp-MarkdownCell .jp-RenderedMarkdown table,
.text_cell_render table,
.rendered_html table {
    direction: rtl;
}
</style>
<div dir=rtl>


📌 توجه کنید که برای نرمال‌سازی، اصلاً لازم نبود حلقه بزنیم یا برای هر ستون جداگانه کد بنویسیم — Broadcasting همه‌ی این کار را در یک خط برایمان انجام داد.

<style>
/* =====================================================================
   persian-rtl-jupyter.css
   هدف: نمایش درست متن فارسی (راست‌به‌چپ) همراه با فرمول‌های ریاضی
   (MathJax) و کد/متن انگلیسی (چپ‌به‌راست) در سلول‌های Markdown نوت‌بوک
   جوپیتر.

   این فایل برای سه محیط پوشش داده شده:
     1) JupyterLab (کلاس‌های .jp-*)
     2) Notebook کلاسیک (کلاس‌های .text_cell_render / .rendered_html)
     3) خروجی nbconvert به HTML و VS Code Jupyter (.rendered_html)
   ===================================================================== */

/* --- 1) جهت پایه‌ی سلول‌های مارک‌داون: راست‌به‌چپ ---
   معادل عملکرد dir="rtl": direction + unicode-bidi:isolate روی کانتینر،
   و direction:rtl روی *همه‌ی* فرزندان (strong, em, li, ...) تا هیچ تگی
   (مثل **بولد**) استایل را گم نکند. */
.jp-MarkdownCell .jp-RenderedMarkdown,
.text_cell_render,
.rendered_html {
    direction: rtl;
    text-align: right;
    unicode-bidi: isolate;
    font-family: "Vazirmatn", "Vazir", "Tahoma", "IRANSans", sans-serif;
    line-height: 1.9;
}

.jp-MarkdownCell .jp-RenderedMarkdown *,
.text_cell_render *,
.rendered_html * {
    direction: rtl;
    unicode-bidi: isolate;
}

/* --- 2) فرمول‌های ریاضی: همیشه چپ‌به‌راست و مستقل از متن اطراف --- */
.jp-MarkdownCell .jp-RenderedMarkdown mjx-container,
.text_cell_render mjx-container,
.rendered_html mjx-container,
.jp-MarkdownCell .jp-RenderedMarkdown .MathJax,
.text_cell_render .MathJax,
.rendered_html .MathJax,
.jp-MarkdownCell .jp-RenderedMarkdown .MathJax_Display,
.text_cell_render .MathJax_Display,
.rendered_html .MathJax_Display {
    direction: ltr;
    unicode-bidi: isolate;               /* فرمول یک "جزیره‌ی" مستقل LTR می‌شود */
    text-align: left;
}

/* فرمول‌های نمایشی (روی خط جدا، $$...$$) وسط‌چین بمانند */
mjx-container[display="true"] {
    text-align: center !important;
}

/* --- 3) کد inline و بلوک‌های کد: چپ‌به‌راست --- */
.jp-MarkdownCell .jp-RenderedMarkdown code,
.jp-MarkdownCell .jp-RenderedMarkdown pre,
.text_cell_render code,
.text_cell_render pre,
.rendered_html code,
.rendered_html pre {
    direction: ltr;
    unicode-bidi: isolate;
    text-align: left;
    font-family: "Fira Code", "Consolas", monospace;
}

/* --- 4) لیست‌ها و جدول‌ها هم راست‌چین شوند --- */
.jp-MarkdownCell .jp-RenderedMarkdown ul,
.jp-MarkdownCell .jp-RenderedMarkdown ol,
.text_cell_render ul,
.text_cell_render ol,
.rendered_html ul,
.rendered_html ol {
    padding-right: 1.5em;
    padding-left: 0;
}

.jp-MarkdownCell .jp-RenderedMarkdown table,
.text_cell_render table,
.rendered_html table {
    direction: rtl;
}
</style>
<div dir=rtl>

## بخش ۵: وقتی Broadcasting شکست می‌خورد — خواندن پیام خطا

📘 نکته‌ی عملی مهم: وقتی shapeها با قانون Broadcasting جور نباشند، پایتون خطا می‌دهد. یاد گرفتن خواندن این خطا، یکی از مهارت‌های روزمره‌ی کدنویسی یادگیری ماشین است.

In [12]:
try:
    bad_matrix = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])   # shape (2, 3)
    bad_vector = torch.tensor([1.0, 2.0])                            # shape (2,)
    result = bad_matrix + bad_vector
except RuntimeError as e:
    print("❌ خطا گرفتیم:", e)
    print("\n📌 چرا؟ چون از راست به چپ مقایسه می‌کنیم: آخرین بُعد ماتریس=3 ولی آخرین بُعد بردار=2 — نه برابرند، نه یکی‌شان ۱ است.")


❌ خطا گرفتیم: The size of tensor a (3) must match the size of tensor b (2) at non-singleton dimension 1

📌 چرا؟ چون از راست به چپ مقایسه می‌کنیم: آخرین بُعد ماتریس=3 ولی آخرین بُعد بردار=2 — نه برابرند، نه یکی‌شان ۱ است.


<style>
/* =====================================================================
   persian-rtl-jupyter.css
   هدف: نمایش درست متن فارسی (راست‌به‌چپ) همراه با فرمول‌های ریاضی
   (MathJax) و کد/متن انگلیسی (چپ‌به‌راست) در سلول‌های Markdown نوت‌بوک
   جوپیتر.

   این فایل برای سه محیط پوشش داده شده:
     1) JupyterLab (کلاس‌های .jp-*)
     2) Notebook کلاسیک (کلاس‌های .text_cell_render / .rendered_html)
     3) خروجی nbconvert به HTML و VS Code Jupyter (.rendered_html)
   ===================================================================== */

/* --- 1) جهت پایه‌ی سلول‌های مارک‌داون: راست‌به‌چپ ---
   معادل عملکرد dir="rtl": direction + unicode-bidi:isolate روی کانتینر،
   و direction:rtl روی *همه‌ی* فرزندان (strong, em, li, ...) تا هیچ تگی
   (مثل **بولد**) استایل را گم نکند. */
.jp-MarkdownCell .jp-RenderedMarkdown,
.text_cell_render,
.rendered_html {
    direction: rtl;
    text-align: right;
    unicode-bidi: isolate;
    font-family: "Vazirmatn", "Vazir", "Tahoma", "IRANSans", sans-serif;
    line-height: 1.9;
}

.jp-MarkdownCell .jp-RenderedMarkdown *,
.text_cell_render *,
.rendered_html * {
    direction: rtl;
    unicode-bidi: isolate;
}

/* --- 2) فرمول‌های ریاضی: همیشه چپ‌به‌راست و مستقل از متن اطراف --- */
.jp-MarkdownCell .jp-RenderedMarkdown mjx-container,
.text_cell_render mjx-container,
.rendered_html mjx-container,
.jp-MarkdownCell .jp-RenderedMarkdown .MathJax,
.text_cell_render .MathJax,
.rendered_html .MathJax,
.jp-MarkdownCell .jp-RenderedMarkdown .MathJax_Display,
.text_cell_render .MathJax_Display,
.rendered_html .MathJax_Display {
    direction: ltr;
    unicode-bidi: isolate;               /* فرمول یک "جزیره‌ی" مستقل LTR می‌شود */
    text-align: left;
}

/* فرمول‌های نمایشی (روی خط جدا، $$...$$) وسط‌چین بمانند */
mjx-container[display="true"] {
    text-align: center !important;
}

/* --- 3) کد inline و بلوک‌های کد: چپ‌به‌راست --- */
.jp-MarkdownCell .jp-RenderedMarkdown code,
.jp-MarkdownCell .jp-RenderedMarkdown pre,
.text_cell_render code,
.text_cell_render pre,
.rendered_html code,
.rendered_html pre {
    direction: ltr;
    unicode-bidi: isolate;
    text-align: left;
    font-family: "Fira Code", "Consolas", monospace;
}

/* --- 4) لیست‌ها و جدول‌ها هم راست‌چین شوند --- */
.jp-MarkdownCell .jp-RenderedMarkdown ul,
.jp-MarkdownCell .jp-RenderedMarkdown ol,
.text_cell_render ul,
.text_cell_render ol,
.rendered_html ul,
.rendered_html ol {
    padding-right: 1.5em;
    padding-left: 0;
}

.jp-MarkdownCell .jp-RenderedMarkdown table,
.text_cell_render table,
.rendered_html table {
    direction: rtl;
}
</style>
<div dir=rtl>


### 🗣️ بحث دو‌نفره — ۲ دقیقه
با نفر کناری‌تان: اگر `bad_vector` را با ۳ عنصر (`[1.0, 2.0, 3.0]`) تعریف می‌کردیم، آیا خطا از بین می‌رفت؟ چرا؟

<style>
/* =====================================================================
   persian-rtl-jupyter.css
   هدف: نمایش درست متن فارسی (راست‌به‌چپ) همراه با فرمول‌های ریاضی
   (MathJax) و کد/متن انگلیسی (چپ‌به‌راست) در سلول‌های Markdown نوت‌بوک
   جوپیتر.

   این فایل برای سه محیط پوشش داده شده:
     1) JupyterLab (کلاس‌های .jp-*)
     2) Notebook کلاسیک (کلاس‌های .text_cell_render / .rendered_html)
     3) خروجی nbconvert به HTML و VS Code Jupyter (.rendered_html)
   ===================================================================== */

/* --- 1) جهت پایه‌ی سلول‌های مارک‌داون: راست‌به‌چپ ---
   معادل عملکرد dir="rtl": direction + unicode-bidi:isolate روی کانتینر،
   و direction:rtl روی *همه‌ی* فرزندان (strong, em, li, ...) تا هیچ تگی
   (مثل **بولد**) استایل را گم نکند. */
.jp-MarkdownCell .jp-RenderedMarkdown,
.text_cell_render,
.rendered_html {
    direction: rtl;
    text-align: right;
    unicode-bidi: isolate;
    font-family: "Vazirmatn", "Vazir", "Tahoma", "IRANSans", sans-serif;
    line-height: 1.9;
}

.jp-MarkdownCell .jp-RenderedMarkdown *,
.text_cell_render *,
.rendered_html * {
    direction: rtl;
    unicode-bidi: isolate;
}

/* --- 2) فرمول‌های ریاضی: همیشه چپ‌به‌راست و مستقل از متن اطراف --- */
.jp-MarkdownCell .jp-RenderedMarkdown mjx-container,
.text_cell_render mjx-container,
.rendered_html mjx-container,
.jp-MarkdownCell .jp-RenderedMarkdown .MathJax,
.text_cell_render .MathJax,
.rendered_html .MathJax,
.jp-MarkdownCell .jp-RenderedMarkdown .MathJax_Display,
.text_cell_render .MathJax_Display,
.rendered_html .MathJax_Display {
    direction: ltr;
    unicode-bidi: isolate;               /* فرمول یک "جزیره‌ی" مستقل LTR می‌شود */
    text-align: left;
}

/* فرمول‌های نمایشی (روی خط جدا، $$...$$) وسط‌چین بمانند */
mjx-container[display="true"] {
    text-align: center !important;
}

/* --- 3) کد inline و بلوک‌های کد: چپ‌به‌راست --- */
.jp-MarkdownCell .jp-RenderedMarkdown code,
.jp-MarkdownCell .jp-RenderedMarkdown pre,
.text_cell_render code,
.text_cell_render pre,
.rendered_html code,
.rendered_html pre {
    direction: ltr;
    unicode-bidi: isolate;
    text-align: left;
    font-family: "Fira Code", "Consolas", monospace;
}

/* --- 4) لیست‌ها و جدول‌ها هم راست‌چین شوند --- */
.jp-MarkdownCell .jp-RenderedMarkdown ul,
.jp-MarkdownCell .jp-RenderedMarkdown ol,
.text_cell_render ul,
.text_cell_render ol,
.rendered_html ul,
.rendered_html ol {
    padding-right: 1.5em;
    padding-left: 0;
}

.jp-MarkdownCell .jp-RenderedMarkdown table,
.text_cell_render table,
.rendered_html table {
    direction: rtl;
}
</style>
<div dir=rtl>


## بخش ۶: کارگاه عملی — روی داده‌ی پروژه‌ی خودتان

با تیم‌تان:

1. یک ماتریس کوچک از داده‌ی فرضی یا واقعی پروژه‌ی خودتان بسازید (چند سطر = چند نمونه، چند ستون = چند ویژگی عددی).
2. با استفاده از Broadcasting (بدون حلقه‌نویسی!) آن را نرمال‌سازی کنید.
3. یک تبدیل دلخواه دیگر هم امتحان کنید — مثلاً افزودن یک بردار «تخفیف» متفاوت به هر ستون، یا ضرب هر سطر در یک ضریب متفاوت (بردار ستونی).
4. نتیجه را برای TA تیم‌تان در ۲ خط توضیح دهید.

In [15]:
# فضای کار تیم شما 👇
# Data source: Dataset.xlsx -> design_matrix (First 4 rows and 4 main features)
my_project_data = torch.tensor([
    [3.636802, 5.0, 94.597161, 3.448059],
    [14.691189, 5.0, 94.597161, 12.193492],
    [4.829448, 5.0, 94.597161, 4.719306],
    [7.805090, 5.0, 94.597161, 1.508667]
])  # TODO: Several rows x columns from project data

# TODO: Compute column-wise mean and std, then normalize the data using broadcasting
col_mean = my_project_data.mean(dim=0)  # Mean per column from dataset
col_std = my_project_data.std(dim=0, unbiased=False)  # Std per column from dataset
normalized_data = (my_project_data - col_mean) / col_std  # Standardized features

# TODO: Test another broadcasting operation (e.g., adding an offset vector per feature)
feature_offset = torch.tensor([1.0, 0.0, 2.0, 0.5])  # Offset vector
adjusted_data = my_project_data + feature_offset  # Broadcasting addition across rows

print("داده‌های اولیه پروژه:\n", my_project_data)
print("\nداده‌های نرمال‌شده:\n", normalized_data)
print("\nداده‌ها پس از اعمال Broadcasting (افزودن افست):\n", adjusted_data)

داده‌های اولیه پروژه:
 tensor([[ 3.6368,  5.0000, 94.5972,  3.4481],
        [14.6912,  5.0000, 94.5972, 12.1935],
        [ 4.8294,  5.0000, 94.5972,  4.7193],
        [ 7.8051,  5.0000, 94.5972,  1.5087]])

داده‌های نرمال‌شده:
 tensor([[-0.9565,     nan,     nan, -0.4988],
        [ 1.6200,     nan,     nan,  1.6615],
        [-0.6785,     nan,     nan, -0.1848],
        [ 0.0150,     nan,     nan, -0.9779]])

داده‌ها پس از اعمال Broadcasting (افزودن افست):
 tensor([[ 4.6368,  5.0000, 96.5972,  3.9481],
        [15.6912,  5.0000, 96.5972, 12.6935],
        [ 5.8294,  5.0000, 96.5972,  5.2193],
        [ 8.8051,  5.0000, 96.5972,  2.0087]])


<style>
/* =====================================================================
   persian-rtl-jupyter.css
   هدف: نمایش درست متن فارسی (راست‌به‌چپ) همراه با فرمول‌های ریاضی
   (MathJax) و کد/متن انگلیسی (چپ‌به‌راست) در سلول‌های Markdown نوت‌بوک
   جوپیتر.

   این فایل برای سه محیط پوشش داده شده:
     1) JupyterLab (کلاس‌های .jp-*)
     2) Notebook کلاسیک (کلاس‌های .text_cell_render / .rendered_html)
     3) خروجی nbconvert به HTML و VS Code Jupyter (.rendered_html)
   ===================================================================== */

/* --- 1) جهت پایه‌ی سلول‌های مارک‌داون: راست‌به‌چپ ---
   معادل عملکرد dir="rtl": direction + unicode-bidi:isolate روی کانتینر،
   و direction:rtl روی *همه‌ی* فرزندان (strong, em, li, ...) تا هیچ تگی
   (مثل **بولد**) استایل را گم نکند. */
.jp-MarkdownCell .jp-RenderedMarkdown,
.text_cell_render,
.rendered_html {
    direction: rtl;
    text-align: right;
    unicode-bidi: isolate;
    font-family: "Vazirmatn", "Vazir", "Tahoma", "IRANSans", sans-serif;
    line-height: 1.9;
}

.jp-MarkdownCell .jp-RenderedMarkdown *,
.text_cell_render *,
.rendered_html * {
    direction: rtl;
    unicode-bidi: isolate;
}

/* --- 2) فرمول‌های ریاضی: همیشه چپ‌به‌راست و مستقل از متن اطراف --- */
.jp-MarkdownCell .jp-RenderedMarkdown mjx-container,
.text_cell_render mjx-container,
.rendered_html mjx-container,
.jp-MarkdownCell .jp-RenderedMarkdown .MathJax,
.text_cell_render .MathJax,
.rendered_html .MathJax,
.jp-MarkdownCell .jp-RenderedMarkdown .MathJax_Display,
.text_cell_render .MathJax_Display,
.rendered_html .MathJax_Display {
    direction: ltr;
    unicode-bidi: isolate;               /* فرمول یک "جزیره‌ی" مستقل LTR می‌شود */
    text-align: left;
}

/* فرمول‌های نمایشی (روی خط جدا، $$...$$) وسط‌چین بمانند */
mjx-container[display="true"] {
    text-align: center !important;
}

/* --- 3) کد inline و بلوک‌های کد: چپ‌به‌راست --- */
.jp-MarkdownCell .jp-RenderedMarkdown code,
.jp-MarkdownCell .jp-RenderedMarkdown pre,
.text_cell_render code,
.text_cell_render pre,
.rendered_html code,
.rendered_html pre {
    direction: ltr;
    unicode-bidi: isolate;
    text-align: left;
    font-family: "Fira Code", "Consolas", monospace;
}

/* --- 4) لیست‌ها و جدول‌ها هم راست‌چین شوند --- */
.jp-MarkdownCell .jp-RenderedMarkdown ul,
.jp-MarkdownCell .jp-RenderedMarkdown ol,
.text_cell_render ul,
.text_cell_render ol,
.rendered_html ul,
.rendered_html ol {
    padding-right: 1.5em;
    padding-left: 0;
}

.jp-MarkdownCell .jp-RenderedMarkdown table,
.text_cell_render table,
.rendered_html table {
    direction: rtl;
}
</style>
<div dir=rtl>


##  تکلیف

**تکلیف قبل از جلسه‌ی بعد:**
1. تمرین‌های `# TODO` این نوت‌بوک را کامل کنید.
2. فصل ۲ *Dive into Deep Learning* (اگر هنوز کامل نخوانده‌اید) را تمام کنید.
3. کمی با NumPy/PyTorch در Colab بازی کنید — چند shape مختلف بسازید و ببینید کدام‌ها با هم broadcast می‌شوند و کدام‌ها نه.

**منابع این جلسه:**
- Zhang et al. — *Dive into Deep Learning*, بخش ۲.۱ تا ۲.۳ (Data Manipulation, Broadcasting)
- Deisenroth et al. — *Mathematics for Machine Learning*, فصل ۲ (مقدمه‌ی جبر خطی، برای یادآوری نماد بردار/ماتریس)

با این جلسه، **بخش اول درس (مقدمه و پایه‌گذاری رویکرد) تمام شد.** از جلسه‌ی بعد وارد بخش دوم (جبر خطی پیشرفته) می‌شویم — جایی که همین تانسورها و Broadcasting امروز، پایه‌ی کار با نرم‌ها، ضرب داخلی و تصویر هندسی خواهند بود.
